# Oncology-DDI — Debug & Integration Notebook
**Purpose:** step-by-step checks for Neo4j, Ollama, AutoGen v0.4 agents, and the pipeline.
Run cells top-to-bottom. Each cell prints clear diagnostics and helpful errors.


In [12]:
import os
os.chdir("C:/Teja/AutoGen_project")
os.getcwd()

'C:\\Teja\\AutoGen_project'

In [13]:
# Cell: Environment & imports
import os
import sys
from pathlib import Path
from pprint import pprint

# Point Python to project root (adjust if needed)
PROJECT_ROOT = Path.cwd()  # or set explicitly: Path("C:/Teja/AutoGen_project")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Python executable:", sys.executable)
print("Working dir:", Path.cwd())


Project root: C:\Teja\AutoGen_project
Python executable: c:\Teja\AutoGen_project\Autogen\Scripts\python.exe
Working dir: C:\Teja\AutoGen_project


In [14]:
# Cell: load .env and show config used by notebook
from dotenv import load_dotenv
load_dotenv(dotenv_path=PROJECT_ROOT / ".env")

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL")

print("NEO4J_URI:", NEO4J_URI)
print("NEO4J_USER:", NEO4J_USER)
print("NEO4J_PASSWORD:", NEO4J_PASSWORD)
print("OLLAMA_MODEL:", OLLAMA_MODEL)


NEO4J_URI: bolt://localhost:7687
NEO4J_USER: neo4j
NEO4J_PASSWORD: Password
OLLAMA_MODEL: llama3.2


In [15]:
# Cell: check Neo4j connectivity and run a small query
from neo4j import GraphDatabase, exceptions

def test_neo4j_connection(uri, user, password):
    try:
        driver = GraphDatabase.driver(uri, auth=(user, password))
        with driver.session() as s:
            res = s.run("RETURN 'connected' AS status")
            print("Neo4j test result:", res.single()["status"])
            # sample counts
            node_count = s.run("MATCH (n) RETURN count(n) AS c").single()["c"]
            rel_count = s.run("MATCH ()-[r]->() RETURN count(r) AS c").single()["c"]
            print(f"Nodes: {node_count}, Relationships: {rel_count}")
        driver.close()
        return True
    except exceptions.AuthError as e:
        print("AuthError:", e)
    except exceptions.ServiceUnavailable as e:
        print("ServiceUnavailable:", e)
    except Exception as e:
        print("Neo4j connection error:", type(e).__name__, e)
    return False

ok = test_neo4j_connection(NEO4J_URI or "neo4j://localhost:7687", NEO4J_USER or "neo4j", NEO4J_PASSWORD or "password")
assert ok, "Neo4j connection failed - start Neo4j Desktop and check .env"


Neo4j test result: connected
Nodes: 15, Relationships: 24


In [16]:
# Cell: run example queries and print tables
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
with driver.session() as s:
    print("\n=== Sample Drugs ===")
    q = "MATCH (d:Drug) RETURN d.name AS name, d.drug_id AS id LIMIT 10"
    for r in s.run(q):
        print(r["name"], "|", r.get("id"))

    print("\n=== Sample Interactions ===")
    q2 = (
        "MATCH (a:Drug)-[r:INTERACTS_WITH]->(b:Drug) "
        "RETURN a.name AS a, b.name AS b, r.severity AS severity, r.mechanism AS mech LIMIT 10"
    )
    for r in s.run(q2):
        print(r["a"], "->", r["b"], "|", r["severity"], "|", r["mech"])
driver.close()



=== Sample Drugs ===
Imatinib | D001
Warfarin | D002
Fluconazole | D003
Doxorubicin | D004
Paclitaxel | D005
Cyclophosphamide | D006
Erlotinib | D007
Gefitinib | D008
Methotrexate | D009
Cisplatin | D010

=== Sample Interactions ===
Cyclophosphamide -> Imatinib | Major | CYP2C9 inhibition
Erlotinib -> Imatinib | Major | CYP3A4 inhibition
Methotrexate -> Imatinib | Major | Folate antagonist
Paclitaxel -> Warfarin | Minor | CYP2C9 inhibition
Warfarin -> Fluconazole | Minor | Folate antagonist
Cyclophosphamide -> Fluconazole | Moderate | DNA damage
Warfarin -> Doxorubicin | Moderate | CYP3A4 inhibition
Cyclophosphamide -> Doxorubicin | Major | CYP3A4 inhibition
Cisplatin -> Paclitaxel | Minor | CYP3A4 inhibition
Erlotinib -> Cyclophosphamide | Major | DNA damage


In [17]:
# Cell: ensure CSVs are present in data/ folder
data_dir = PROJECT_ROOT / "data"
expected = ["drugs.csv", "interactions.csv", "indications.csv", "conditions.csv"]
missing = [f for f in expected if not (data_dir / f).exists()]
print("data/ exists:", data_dir.exists())
if missing:
    print("Missing CSVs:", missing)
else:
    print("All CSV files present.")
    # Show first lines for quick sanity
    import csv
    for f in expected:
        print(f"\n--- {f} (first 3 rows) ---")
        with open(data_dir / f, newline="", encoding="utf-8") as fh:
            reader = csv.reader(fh)
            for i, row in enumerate(reader):
                print(row)
                if i >= 2:
                    break


data/ exists: True
All CSV files present.

--- drugs.csv (first 3 rows) ---
['drug_id', 'name', 'atc_codes']
['D001', 'Imatinib', 'L01XE01']
['D002', 'Warfarin', 'B01AA03']

--- interactions.csv (first 3 rows) ---
['drug_a_id', 'drug_b_id', 'mechanism', 'severity', 'evidence_pubmed', 'evidence_label_url']
['D007', 'D001', 'CYP3A4 inhibition', 'Major', 'PMID:234567', 'https://labels.fda.gov/warfarin']
['D009', 'D001', 'Folate antagonist', 'Major', 'PMID:567890', 'https://labels.fda.gov/doxorubicin']

--- indications.csv (first 3 rows) ---
['drug_id', 'condition']
['D001', 'Chronic Myeloid Leukemia']
['D002', 'Chronic Myeloid Leukemia']

--- conditions.csv (first 3 rows) ---
['id', 'name']
['C001', 'Chronic Myeloid Leukemia']
['C002', 'Breast Cancer']


In [18]:
# Cell: check ollama CLI from Python (requires ollama in PATH)
import subprocess, shlex

def run_cmd(cmd):
    try:
        out = subprocess.check_output(shlex.split(cmd), stderr=subprocess.STDOUT, text=True)
        return out.strip()
    except subprocess.CalledProcessError as e:
        return f"ERROR (exit {e.returncode}): {e.output}"

print("ollama version / list:")
print(run_cmd("ollama --version"))
print(run_cmd("ollama list")[:1000])  # print up to 1000 chars

print("\nIf listed models do not include", OLLAMA_MODEL, "run: ollama pull", OLLAMA_MODEL)


ollama version / list:


FileNotFoundError: [WinError 2] The system cannot find the file specified

In [19]:
# Cell: verify ollama python package available and can list models via HTTP client
try:
    import ollama
    print("Ollama python package imported, version:", getattr(ollama, "__version__", "unknown"))
    # the high-level client usage can vary: attempt a small ping
    try:
        client = ollama.Client()  # newer versions might use different class names
        print("Ollama.Client() created:", type(client))
    except Exception as e:
        print("Could not instantiate ollama.Client():", e)
except Exception as e:
    print("ollama python package not installed in this kernel:", type(e).__name__, e)


Ollama python package imported, version: unknown
Ollama.Client() created: <class 'ollama._client.Client'>


In [20]:
# Cell: test LLMAgentV04.extract_slots (requires AutoGen + autogen_ext[ollama] installed)
from agents.llm_agent_autogen import LLMAgentV04
llm = LLMAgentV04()

test_questions = [
    "Does Fluconazole interact with Warfarin?",
    "Is Warfarin contraindicated for Thrombosis?",
    "What happens if Imatinib is given with Fluconazole?"
]

for q in test_questions:
    try:
        slots = llm.extract_slots(q)
        print("\nQ:", q)
        print("Slots:", slots)
    except Exception as e:
        print("LLM extraction error for Q:", q, "->", type(e).__name__, e)
        # show hint
        import traceback; traceback.print_exc()


LLM extraction error for Q: Does Fluconazole interact with Warfarin? -> RuntimeError This event loop is already running
LLM extraction error for Q: Is Warfarin contraindicated for Thrombosis? -> RuntimeError This event loop is already running
LLM extraction error for Q: What happens if Imatinib is given with Fluconazole? -> RuntimeError This event loop is already running


Traceback (most recent call last):
  File "C:\Users\venka\AppData\Local\Temp\ipykernel_10332\2671660669.py", line 13, in <module>
    slots = llm.extract_slots(q)
            ^^^^^^^^^^^^^^^^^^^^
  File "C:\Teja\AutoGen_project\agents\llm_agent_autogen.py", line 49, in extract_slots
    return _def(self._extract_slots_async(user_text))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Teja\AutoGen_project\agents\llm_agent_autogen.py", line 10, in <lambda>
    _def = lambda coro: asyncio.get_event_loop().run_until_complete(coro)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\venka\AppData\Local\Programs\Python\Python311\Lib\asyncio\base_events.py", line 626, in run_until_complete
    self._check_running()
  File "C:\Users\venka\AppData\Local\Programs\Python\Python311\Lib\asyncio\base_events.py", line 586, in _check_running
    raise RuntimeError('This event loop is already running')
RuntimeError: This event loop is already runni

In [26]:
# Cell: test KGClient wrapper and the exact Cypher used by pipeline
from agents.kg_agent import KGClient, CY_DDI, CY_CONTRA
kg = KGClient()

# Run the same Cypher the pipeline uses for a known pair
params = {"drug1": "Fluconazole", "drug2": "Warfarin"}
rows = kg.query(CY_DDI, params)
print("DDI query rows (Fluconazole + Warfarin):", rows)

# Contra example
rows_contra = kg.query(CY_CONTRA, {"drug": "Warfarin", "cond": "Thrombosis"})
print("Contra query rows:", rows_contra)


DDI query rows (Fluconazole + Warfarin): []
Contra query rows: []


In [22]:
# Cell: test pipeline answers
from agents.pipeline import PIPELINE

questions = [
    "Does Fluconazole interact with Warfarin?",
    "Is Warfarin contraindicated for Thrombosis?",
    "What happens if Imatinib is given with Fluconazole?"
]

for q in questions:
    try:
        ans = PIPELINE.answer(q)
        print("\nQ:", q)
        print("A:", ans)
    except Exception as e:
        print("Pipeline error for Q:", q, "->", type(e).__name__, e)
        import traceback; traceback.print_exc()


Pipeline error for Q: Does Fluconazole interact with Warfarin? -> RuntimeError This event loop is already running
Pipeline error for Q: Is Warfarin contraindicated for Thrombosis? -> RuntimeError This event loop is already running
Pipeline error for Q: What happens if Imatinib is given with Fluconazole? -> RuntimeError This event loop is already running


Traceback (most recent call last):
  File "C:\Users\venka\AppData\Local\Temp\ipykernel_10332\2916238575.py", line 12, in <module>
    ans = PIPELINE.answer(q)
          ^^^^^^^^^^^^^^^^^^
  File "C:\Teja\AutoGen_project\agents\pipeline.py", line 12, in answer
    slots = self.llm.extract_slots(question)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Teja\AutoGen_project\agents\llm_agent_autogen.py", line 49, in extract_slots
    return _def(self._extract_slots_async(user_text))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Teja\AutoGen_project\agents\llm_agent_autogen.py", line 10, in <lambda>
    _def = lambda coro: asyncio.get_event_loop().run_until_complete(coro)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\venka\AppData\Local\Programs\Python\Python311\Lib\asyncio\base_events.py", line 626, in run_until_complete
    self._check_running()
  File "C:\Users\venka\AppData\Local\Programs\Python\Python311\Lib\asynci

In [24]:
# Cell: run the simple evaluation script and show accuracy
from eval.evaluate import None as skip  # if your evaluate script is function-style, import; otherwise run as subprocess

# Simpler: run the evaluate.py script as a process to see console output
import subprocess, shlex, sys
cmd = f"{sys.executable} eval/evaluate.py"
print("Running:", cmd)
print(subprocess.check_output(shlex.split(cmd), text=True, stderr=subprocess.STDOUT))


SyntaxError: invalid syntax (2019342507.py, line 2)

In [25]:
# Cell: Final integration validator (quick)
def validate_all():
    ok = {}
    ok['neo4j'] = test_neo4j_connection(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
    ok['csvs'] = all((PROJECT_ROOT / "data" / f).exists() for f in ["drugs.csv","interactions.csv","indications.csv","conditions.csv"])
    ok['ollama_cli'] = "llama" in (run_cmd("ollama list") if 'run_cmd' in globals() else "")  # heuristic
    try:
        llm = LLMAgentV04()
        ok['llm_extract'] = bool(llm.extract_slots("Does Fluconazole interact with Warfarin?"))
    except Exception as e:
        ok['llm_extract'] = False
        print("LLM extract error:", e)
    try:
        rows = KGClient().query(CY_DDI, {"drug1":"Fluconazole","drug2":"Warfarin"})
        ok['kg_query'] = len(rows) > 0
    except Exception as e:
        ok['kg_query'] = False
        print("KG query error:", e)
    print("Validation summary:")
    pprint(ok)
    return ok

validate_all()


Neo4j test result: connected
Nodes: 15, Relationships: 24


FileNotFoundError: [WinError 2] The system cannot find the file specified